# 02 — Exploratory Data Analysis

**NIFTY 50 Next-Day Direction Prediction**

This notebook covers:
1. Feature correlation heatmap (verify no |corr| > 0.85)
2. Feature distributions by target class
3. Target class balance analysis
4. Feature-target correlations
5. Time-series plots of key features (gap_x_vol, vol_5d, volume_ratio_20d, close_vs_low)

Final 12 features include 5 binary direction encodings (`ret_1d_sign`, `open_gap_sign`, `vix_above_ma20`, `bn_ret_1d_sign`, `vix_ret_1d_sign`) and 7 continuous features. Binary encoding of direction signals was the key research finding — LightGBM cannot reliably split continuous features at exactly zero.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent) if 'notebooks' in str(Path.cwd()) else str(Path.cwd()))

from src.features import build_features, FEATURE_COLS

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12
sns.set_palette('RdBu_r')

X, y = build_features()
print(f'X shape: {X.shape}')
print(f'y distribution: up={int(y.sum())}, down={int(len(y)-y.sum())}')

## 1. Feature Correlation Heatmap

In [ ]:
corr = X.corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', center=0,
    cmap='RdBu_r', vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.7, 'label': 'Pearson Correlation'},
    ax=ax
)
ax.set_title('Feature-Feature Correlation Matrix (Final 12 Features)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../report/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Verify constraint
abs_corr = corr.abs()
np.fill_diagonal(abs_corr.values, 0)
max_corr = abs_corr.max().max()
pair = abs_corr.stack().idxmax()
print(f'\nMax pairwise |correlation|: {max_corr:.4f} ({pair[0]} vs {pair[1]})')
print(f'Constraint satisfied: {max_corr < 0.85}')

## 2. Feature Distributions (Histograms by Target Class)

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(18, 16))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    ax = axes[i]
    for label, color, name in [(0, '#e74c3c', 'Down'), (1, '#2ecc71', 'Up')]:
        ax.hist(X.loc[y == label, col], bins=40, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylabel('Density')

fig.suptitle('Feature Distributions by Target Class', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../report/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Target Class Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
counts = y.value_counts().sort_index()
colors = ['#e74c3c', '#2ecc71']
axes[0].bar(['Down (0)', 'Up (1)'], counts.values, color=colors, edgecolor='white', linewidth=2)
axes[0].set_title('Target Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, f'{v}\n({v/len(y)*100:.1f}%)', ha='center', fontsize=12, fontweight='bold')

# Rolling target rate
rolling_rate = y.rolling(63).mean()
axes[1].plot(X.index, rolling_rate, color='#3498db', linewidth=2)
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('Rolling 63-Day Up Probability', fontsize=14, fontweight='bold')
axes[1].set_ylabel('P(Up)')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.savefig('../report/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature-Target Correlations

In [ ]:
target_corr = X.corrwith(y).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Pearson Correlation with Target')
ax.set_title('Feature-Target Correlations', fontsize=14, fontweight='bold')
for i, (idx, v) in enumerate(target_corr.items()):
    ax.text(v + 0.002 * np.sign(v), i, f'{v:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../report/feature_target_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Time-Series Plots of Key Features

In [ ]:
key_features = ['gap_x_vol', 'vol_5d', 'volume_ratio_20d', 'close_vs_low']

fig, axes = plt.subplots(len(key_features), 1, figsize=(16, 12), sharex=True)
for i, feat in enumerate(key_features):
    axes[i].plot(X.index, X[feat], color='#3498db', linewidth=1, alpha=0.8)
    axes[i].set_ylabel(feat)
    axes[i].set_title(feat, fontsize=12, fontweight='bold')

axes[-1].set_xlabel('Date')
fig.suptitle('Time-Series of Key Features', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../report/feature_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary Statistics

In [ ]:
stats = X.describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]
stats['skew'] = X.skew()
stats['kurtosis'] = X.kurtosis()
print(stats.to_string())

## 7. Key Findings

1. **No correlation exceeds 0.85** — max is ret_1d_abs vs high_low_range at 0.641, well under the 0.85 constraint
2. **Target is near-balanced** — approximately 52% up, 48% down
3. **5 of 12 features are binary** — ret_1d_sign, open_gap_sign, vix_above_ma20, bn_ret_1d_sign, vix_ret_1d_sign. Binary encoding captures the lag-1 direction autocorrelation (+0.107) that is the strongest predictive signal
4. **Starter feature audit**: `ma5_smooth_signal` rejected for look-ahead leakage; `ret_zscore` rejected for perfect collinearity with `ret_1d`
5. **Strongest target predictors**: ret_1d_sign (+0.107 autocorrelation), gap_x_vol (55.6% solo OOF accuracy), bn_ret_1d_sign (cross-asset agreement) — all individually weak, as expected for daily equity direction
6. **Feature matrix** is 959 × 12, clean with zero NaN values
7. **Top interaction**: gap_x_vol captures overnight gap magnitude relative to volatility regime — large gaps in calm markets carry more directional signal